# Load and Parse Data Example

Build efficient input pipelines for loading data, with Grain. JAX itself
does not load data: Grain is the data loading library of the JAX ecosystem.
It reads records from a data source, and chains transformations (shuffle,
map, batch, ...) that are run in background threads, so that the next
batches are ready when the model needs them.

- Numpy Arrays
- CSV file
- Images
- Custom data from a Python class

For more information about creating and loading ArrayRecord files (the
record format used by Grain), see 'array_records.py'.

- Author: Aymeric Damien
- Project: [https://github.com/aymericdamien/JAX-Examples/](https://github.com/aymericdamien/JAX-Examples/)

In [1]:
import csv
import os
import shutil
import string
import sys
import tarfile

import grain
import numpy as np
from PIL import Image

# Make the shared helpers of the 'common' folder importable.
sys.path.append(os.path.join(os.getcwd(), "..", ".."))
from common.datasets import DATA_DIR, download

### Load Numpy Arrays

Build a data pipeline over numpy arrays.

In [2]:
# Create a toy dataset (even and odd numbers, with respective labels of 0 and 1).
evens = np.arange(0, 100, step=2, dtype=np.int32)
evens_label = np.zeros(50, dtype=np.int32)
odds = np.arange(1, 100, step=2, dtype=np.int32)
odds_label = np.ones(50, dtype=np.int32)
# Concatenate arrays
features = np.concatenate([evens, odds])
labels = np.concatenate([evens_label, odds_label])

# Load the data with Grain: any Python sequence (here a list of (feature, label)
# pairs) can be the source of a dataset.
data = grain.MapDataset.source(list(zip(features, labels)))
# Shuffle data (with a seed, for reproducibility).
data = data.shuffle(seed=0)
# Refill data indefinitely (re-shuffled at every epoch).
data = data.repeat()
# Batch data (aggregate records together).
data = data.batch(batch_size=4)

In [3]:
# A 'MapDataset' can be indexed and sliced like a list: take the first 5 batches.
for batch_x, batch_y in data[:5]:
    print(batch_x, batch_y)

[98 67 20 75] [0 1 0 1]
[78 39 91 18] [0 1 1 0]
[25 40  9 14] [1 0 1 0]
[46 26 99 70] [0 0 1 0]
[32  0 66  7] [0 0 0 1]


In [4]:
# Note: If you are planning on calling multiple time,
# you can use the iterator way. 'to_iter_dataset' also prefetches batches
# (pre-load batches in background threads for faster consumption).
ite_data = iter(data.to_iter_dataset(grain.ReadOptions(num_threads=2, prefetch_buffer_size=2)))
for i in range(5):
    batch_x, batch_y = next(ite_data)
    print(batch_x, batch_y)

for i in range(5):
    batch_x, batch_y = next(ite_data)
    print(batch_x, batch_y)

[98 67 20 75] [0 1 0 1]
[78 39 91 18] [0 1 1 0]
[25 40  9 14] [1 0 1 0]
[46 26 99 70] [0 0 1 0]
[32  0 66  7] [0 0 0 1]
[71 19 50 88] [1 1 0 0]
[43 80 30 28] [1 0 0 0]
[41 48 21 92] [1 0 1 0]
[17 62 65  1] [1 0 1 1]
[38 37 23 96] [0 1 1 0]


### Load CSV files

Build a data pipeline from features stored in a CSV file. For this example, Titanic dataset will be used as a toy dataset stored in CSV format.

#### Titanic Dataset

survived|pclass|name|sex|age|sibsp|parch|ticket|fare
--------|------|----|---|---|-----|-----|------|----
1|1|"Allen, Miss. Elisabeth Walton"|female|29|0|0|24160|211.3375
1|1|"Allison, Master. Hudson Trevor"|male|0.9167|1|2|113781|151.5500
0|1|"Allison, Miss. Helen Loraine"|female|2|1|2|113781|151.5500
0|1|"Allison, Mr. Hudson Joshua Creighton"|male|30|1|2|113781|151.5500
...|...|...|...|...|...|...|...|...

In [5]:
# Download Titanic dataset (in csv format).
titanic_path = download(
    ["https://raw.githubusercontent.com/tflearn/tflearn.github.io/master/resources/titanic_dataset.csv"],
    "titanic_dataset.csv")

In [6]:
# Load Titanic dataset.
# Original features: survived,pclass,name,sex,age,sibsp,parch,ticket,fare
# Select specific columns: survived,pclass,name,sex,age,fare
column_to_use = [0, 1, 2, 3, 4, 8]
record_types = [int, int, str, str, float, float]

# Load the whole dataset file, and parse each line with Python's csv module.
records = []
with open(titanic_path) as f:
    reader = csv.reader(f, skipinitialspace=True)
    # Skip header.
    next(reader)
    for row in reader:
        # Keep the selected columns, converted to their type.
        record = tuple(t(row[c]) for c, t in zip(column_to_use, record_types))
        records.append(record)

data = grain.MapDataset.source(records)
# Shuffle data.
data = data.shuffle(seed=0)
# Refill data indefinitely.
data = data.repeat()
# Batch data (aggregate records together).
data = data.batch(batch_size=2)

In [7]:
for survived, pclass, name, sex, age, fare in data[:1]:
    print(survived)
    print(pclass)
    print(name)
    print(sex)
    print(age)
    print(fare)

[1 1]
[1 1]
['Goldenberg, Mrs. Samuel L (Edwiga Grabowska)' 'Goldenberg, Mr. Samuel L']
['female' 'male']
[ 0. 49.]
[89.1042 89.1042]


### Load Images

Build a data pipeline by loading images from disk. For this example, Oxford Flowers dataset will be used.

In [8]:
# Download Oxford 17 flowers dataset
archive_path = download(["https://www.robots.ox.ac.uk/~vgg/data/flowers/17/17flowers.tgz"],
                        "17flowers.tgz")
# Extract archive (images are stored in a 'jpg' folder).
flowers_dir = os.path.join(DATA_DIR, "17flowers")
if not os.path.exists(flowers_dir):
    # Extract to a temporary folder, renamed once complete: an interrupted
    # extraction is then not mistaken for a complete one.
    shutil.rmtree(flowers_dir + ".tmp", ignore_errors=True)
    with tarfile.open(archive_path) as t:
        t.extractall(flowers_dir + ".tmp", filter="data")
    os.rename(flowers_dir + ".tmp", flowers_dir)

In [9]:
# Write a dataset file, listing every image with its label: the 17 flower
# species are stored in order, 80 images each.
with open(os.path.join(flowers_dir, "jpg", "dataset.csv"), 'w') as f:
    c = 0
    for i in range(1360):
        image_path = os.path.join(flowers_dir, "jpg", "image_%04i.jpg" % (i + 1))
        f.write("%s,%i\n" % (image_path, c))
        if (i + 1) % 80 == 0:
            c += 1

In [10]:
# Load Images
with open(os.path.join(flowers_dir, "jpg", "dataset.csv")) as f:
    dataset_file = f.read().splitlines()

# Load the whole dataset file, and slice each line.
data = grain.MapDataset.source(dataset_file)
# Shuffle data.
data = data.shuffle(seed=0)
# Refill data indefinitely.
data = data.repeat()

# Load and pre-process images.
def load_image(path):
    # Read image from path, and decode the jpeg image (with Pillow).
    image = Image.open(path).convert('RGB')
    # Resize images to a common size of 256x256.
    image = image.resize((256, 256))
    # Convert to an array of float32 values in [0, 255].
    image = np.array(image, np.float32)
    # Rescale values to [-1, 1].
    image = image / 127.5 - 1.
    return image

# Decode each line from the dataset file.
def parse_records(line):
    # File is in csv format: "image_path,label_id".
    image_path, image_label = line.split(',')
    # Apply the function to load images.
    image = load_image(image_path)
    return image, int(image_label)

# Use 'map' to apply the above functions to every record.
data = data.map(parse_records)

# Batch data (aggregate images-array together).
data = data.batch(batch_size=2)
# Prefetch batch: images are loaded and decoded in 4 parallel threads.
data = data.to_iter_dataset(grain.ReadOptions(num_threads=4, prefetch_buffer_size=4))

In [11]:
batch_x, batch_y = next(iter(data))
print("images:", batch_x.shape, batch_x.dtype, "min: %f, max: %f" % (batch_x.min(), batch_x.max()))
print("labels:", batch_y)

images: (2, 256, 256, 3) float32 min: -1.000000, max: 1.000000
labels: [1 1]


### Load custom data from a Python class

Grain does not read Python generators: to shuffle the data (or to resume an interrupted
training), it needs random access to the records. So custom data comes from a 'data source':
any Python object with a `__len__` (number of records) and a `__getitem__` (return the record
at a given index) method, which can load or generate each record on the fly.

In [12]:
# Create a dummy data source.
class RandomFeatures:
    # Total number of records.
    def __len__(self):
        return 1000

    # Generate the record number 'index'.
    def __getitem__(self, index):
        # A random number generator seeded with the index: the same index
        # always returns the same record.
        rng = np.random.default_rng(index)
        # Return a random string, a random vector, and a random int.
        random_string = ''.join(rng.choice(list(string.ascii_letters), size=4))
        return random_string, rng.uniform(size=4).astype(np.float32), int(rng.integers(0, 10))

In [13]:
# Load the data source with Grain.
data = grain.MapDataset.source(RandomFeatures())
# Shuffle data.
data = data.shuffle(seed=0)
# Refill data indefinitely.
data = data.repeat()
# Batch data (aggregate records together).
data = data.batch(batch_size=4)

In [14]:
# Display data.
for batch_str, batch_vector, batch_int in data[:5]:
    print(batch_str, batch_vector, batch_int)

['OYvA' 'BYcA' 'nbNe' 'WSGL'] [[0.54418546 0.7801187  0.30260202 0.7968523 ]
 [0.36084908 0.5762304  0.598123   0.23794304]
 [0.06371503 0.7690869  0.6220628  0.899588  ]
 [0.00636697 0.71242434 0.7906714  0.9454319 ]] [3 5 5 4]
['lQfl' 'cEJB' 'xCcV' 'yxYk'] [[0.88541895 0.7230738  0.10245323 0.94639164]
 [0.8649617  0.7347604  0.47658485 0.8166962 ]
 [0.22616054 0.38066646 0.41035008 0.476755  ]
 [0.44052345 0.51560587 0.5589658  0.14662573]] [9 6 1 9]
['dVqp' 'iPvc' 'EBqm' 'RIeU'] [[0.25376374 0.14586005 0.19089134 0.20657578]
 [0.7379483  0.7395985  0.970987   0.10087088]
 [0.878268   0.40788358 0.196836   0.4822614 ]
 [0.34503612 0.52718806 0.6433215  0.08524746]] [2 5 5 9]
['pxUB' 'lMus' 'foUY' 'Zzeu'] [[0.7492244  0.17923026 0.579031   0.4761606 ]
 [0.91705364 0.87607974 0.18265282 0.10801628]
 [0.9766844  0.18238322 0.41021606 0.8962021 ]
 [0.11221106 0.87793744 0.01374216 0.38031468]] [4 4 8 5]
['zsGz' 'QLht' 'qOXr' 'nAxG'] [[0.8774416  0.02621511 0.54004806 0.47466847]
 [0.028